## Fast Food Density & Income Pipeline
CSE 3104 — Data Manipulation & Management: Final Project 

Connects ACS 5-Year Census income data to McDonald's locations by zip code.

In [24]:
# Libraries we need
import pandas as pd
import numpy as np
import kagglehub
import os


In [ ]:
# load the census income file, skip the extra header row Census adds
census = pd.read_csv("source_data/ACSDT5Y2024.B19013-Data.csv", skiprows=[1])

census["zip_code"] = census["NAME"].str.extract(r"(\d{5})") # pull just the 5 digit zip
census["median_income"] = pd.to_numeric(census["B19013_001E"], errors="coerce") # make sure income is a number
census = census[["zip_code", "median_income"]].copy() # only keep what we need

census.head(10)

,zip_code,median_income
0,00601,19454.0
1,00602,21420.0
2,00603,20933.0
3,00606,20992.0
4,00610,24496.0
5,00611,20360.0
6,00612,24488.0
7,00616,21846.0
8,00617,25013.0
9,00622,26585.0


In [26]:
# some zips are missing income data, fill them in using the average of nearby zips in the same state
census["state_prefix"] = census["zip_code"].str[:2] # first 2 digits = state

missing_before = census["median_income"].isna().sum()

census["median_income"] = census.groupby("state_prefix")["median_income"].transform(
    lambda x: x.fillna(x.mean())
)

missing_after = census["median_income"].isna().sum()
print(f"Imputed {missing_before - missing_after} values ({missing_before} → {missing_after} missing)")

census = census.drop(columns="state_prefix") # dont need this anymore
census.head(3)

Imputed 3358 values (3358 → 0 missing)


,zip_code,median_income
0,00601,19454.0
1,00602,21420.0
2,00603,20933.0


In [ ]:
# load mcdonalds locations — this file already has latitude and longitude
mcdonalds = pd.read_csv("source_data/mcdonalds_locations.csv")

# clean up the zip column so it matches the census format
mcdonalds = mcdonalds.rename(columns={"zipcode": "zip_code"})
mcdonalds["zip_code"] = mcdonalds["zip_code"].astype(str).str.zfill(5).str[:5]

# keep lat/lon so Julieat can use this in ArcGIS
mcdonalds = mcdonalds[["zip_code", "city", "state", "latitude", "longitude"]].copy()

print(f"Loaded {len(mcdonalds):,} McDonald's locations")
print(f"Missing coordinates: {mcdonalds['latitude'].isna().sum()} rows")
mcdonalds.head(3)

Loaded 13,418 McDonald's locations
Missing coordinates: 0 rows


,zip_code,city,state,latitude,longitude
0,33040,Key West,FL,24.571897,-81.756906
1,33050,Marathon,FL,24.716418,-81.073326
2,33070,Tavernier,FL,25.004612,-80.522609


In [ ]:
# merge mcdonalds locations with census income so we have income + coordinates in one file
mcdonalds_geo = mcdonalds.merge(census, on="zip_code", how="left")

# drop rows with no coordinates — cant map those in arcgis
mcdonalds_geo = mcdonalds_geo.dropna(subset=["latitude", "longitude"])

# save it — this is the file Julieat can bring straight into ArcGIS
mcdonalds_geo.to_csv("outputs/mcdonalds_locations_gis.csv", index=False)
print(f"Saved {len(mcdonalds_geo):,} mappable McDonald's locations to outputs/mcdonalds_locations_gis.csv")
mcdonalds_geo.head(3)

Saved 13,418 mappable McDonald's locations to mcdonalds_locations_gis.csv


,zip_code,city,state,latitude,longitude,median_income
0,33040,Key West,FL,24.571897,-81.756906,86586.0
1,33050,Marathon,FL,24.716418,-81.073326,94898.0
2,33070,Tavernier,FL,25.004612,-80.522609,75436.0


In [29]:
# download shake shack locations from kaggle
path = kagglehub.dataset_download("jeffreybraun/shake-shack-restaurant-locations")

# find the csv in the downloaded folder
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
shakeshack = pd.read_csv(os.path.join(path, csv_files[0]))

print(f"Loaded {len(shakeshack):,} Shake Shack locations")
shakeshack.head(3)

Loaded 193 Shake Shack locations


,state,name,address
0,Alabama,"Birmingham, AL","200 Summit Blvd. Suite 100, Birmingham, Alabam..."
1,Arizona,"Fashion Square, AZ",7014 E. Camelback Road
2,Arizona,"Kierland Commons, AZ",15030 N Scottsdale Road


In [30]:
# count how many mcdonalds are in each zip code
store_counts = (
    mcdonalds
    .groupby("zip_code")
    .size()
    .reset_index(name="store_count")
)

print(f"Unique zips with a McDonald's: {len(store_counts):,}")
store_counts.head(3)

Unique zips with a McDonald's: 9,175


,zip_code,store_count
0,01001,1
1,01007,1
2,01008,2


In [31]:
# combine income and store count into one table
merged = census.merge(store_counts, on="zip_code", how="left")
merged["store_count"] = merged["store_count"].fillna(0).astype(int) # zips with no mcdonalds get 0

matched = (merged["store_count"] > 0).sum()
unmatched = len(store_counts) - matched

print(f"Zips matched to a McDonald's: {matched:,}")
print(f"McDonald's locations outside Census zips: {unmatched:,}")

# out of all the mcdonalds zips, how many actually matched to census data
match_rate = (matched / len(store_counts)) * 100
print(f"Match rate: {match_rate:.2f}%")

merged.head(5)

Zips matched to a McDonald's: 9,082
McDonald's locations outside Census zips: 93
Match rate: 98.99%


,zip_code,median_income,store_count
0,00601,19454.0,0
1,00602,21420.0,0
2,00603,20933.0,0
3,00606,20992.0,0
4,00610,24496.0,0


In [ ]:
# load population data so we can calculate store density later
demographics = pd.read_csv("source_data/ACSDP5Y2024.DP05-Data.csv", skiprows=[1])
demographics["zip_code"] = demographics["NAME"].str.extract(r"(\d{5})")
demographics["total_population"] = pd.to_numeric(demographics["DP05_0001E"], errors="coerce")
demographics = demographics[["zip_code", "total_population"]]
demographics.head(100)

/var/folders/v8/b8h2h0rd3t5fl9n725gtjk0h0000gn/T/ipykernel_20813/1211330373.py:2: DtypeWarning: Columns (3,5,7,11,17,19,21,23,25,31,39,43,49,51,53,55,59,61,63,67,165,179,181,191) have mixed types. Specify dtype option on import or set low_memory=False.
  demographics = pd.read_csv("ACSDP5Y2024.DP05-Data.csv", skiprows=[1])


,zip_code,total_population
0,00601,16669
1,00602,37233
2,00603,48448
3,00606,5163
4,00610,25357
...,...,...
95,00909,5633
96,00911,7297
97,00912,5845
98,00913,6943


In [ ]:
# save the final table to a csv
OUT_PATH = "outputs/National_Income_FastFood_Merged.csv"
merged.to_csv(OUT_PATH, index=False)
print(f"Saved {len(merged):,} rows to {OUT_PATH}")

Saved 33,772 rows to National_Income_FastFood_Merged.csv
